# Lab 4

## Task 1

**SUMMARY:** *Through the previous labs, we've gained a lot of tools to attack unknown embedded devices: SPA, DPA, CPA, trace resynchronization, and more. In this lab, we'll be using some of those techniques to break a more realistic target: a bootloader. Note that there are two versions of this lab. In this one (Lab A), we'll proceed with a lot of knowledge about how the bootloader is working. In a real scenario, we may or may not be able to get that information from a datasheet. In Lab B, we'll do some reverse engineering work to figure this stuff out on our own. It's up to you whether you want to run this lab or Lab B first!*

**LEARNING OUTCOMES:**

* Extending AES128 attacks to AES256
* Using DPA to recover an initialization vector
* Using a timing attack (via power analysis) to recover a signature

This tutorial will take you through a complete attack on an encrypted bootloader using AES-256. This demonstrates how to use side-channel power analysis on practical systems, along with discussing how to perform analysis with different Analyzer models.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'

## Background

In the world of microcontrollers, a bootloader is a special piece of firmware that is made to let the user upload new programs into memory. This is especially useful for devices with complex code that may need to be patched or otherwise updated in the future - a bootloader makes it possible for the user to upload a patched version of the firmware onto the micro. The bootloader receives information from a communication line (a USB port, serial port, ethernet port, WiFi connection, etc...) and stores this data into program memory. Once the full firmware has been received, the micro can happily run its updated code.

There is one big security issue to worry about with bootloaders. A company may want to stop their customers from writing their own firmware and uploading it onto the micro. For example, this might be for protection reasons - hackers might be able to access parts of the device that weren't meant to be accessed. One way of stopping this is to add encryption. The company can add their own secret signature to the firmware code and encrypt it with a secret key. Then, the bootloader can decrypt the incoming firmware and confirm that the incoming firmware is correctly signed. Users will not know the secret key or the signature tied to the firmware, so they won't be able to "fake" their own.

This tutorial will work with a simple AES-256 bootloader. The victim will receive data through a serial connection, decrypt the command, and confirm that the included signature is correct. Then, it will only save the code into memory if the signature check succeeded. To make this system more robust against attacks, the bootloader will use cipher-block chaining (CBC mode). Our goal is to find the secret key and the CBC initialization vector so that we could successfully fake our own firmware.

### Bootloader Communications Protocol

The bootloader's communications protocol operates over a serial port at 38400 baud rate. The bootloader is always waiting for new data to be sent in this example; in real life one would typically force the bootloader to enter through a command sequence.

Commands sent to the bootloader look as follows:

```
       |<-------- Encrypted block (16 bytes) ---------->|
       |                                                |
+------+------+------+------+------+------+ .... +------+------+------+
| 0x00 |    Signature (4 Bytes)    |  Data (12 Bytes)   |   CRC-16    |
+------+------+------+------+------+------+ .... +------+------+------+
```

This frame has four parts:

* `0x00`: 1 byte of fixed header
* Signature: A secret 4 byte constant. The bootloader will confirm that this signature is correct after decrypting the frame.
* Data: 12 bytes of the incoming firmware. This system forces us to send the code 12 bytes at a time; more complete bootloaders may allow longer variable-length frames.
* CRC-16: A 16-bit checksum using the CRC-CCITT polynomial (0x1021). The LSB of the CRC is sent first, followed by the MSB. The bootloader will reply over the serial port, describing whether or not this CRC check was valid.

As described in the diagram, the 16 byte block is not sent as plaintext. Instead, it is encrypted using AES-256 in CBC mode. This encryption method will be described in the next section.

The bootloader responds to each command with a single byte indicating if the CRC-16 was OK or not:

```
            +------+
CRC-OK:     | 0xA1 |
            +------+

            +------+
CRC Failed: | 0xA4 |
            +------+
```
Then, after replying to the command, the bootloader veries that the signature is correct. If it matches the expected manufacturer's signature, the 12 bytes of data will be written to flash memory. Otherwise, the data is discarded.

### Details of AES-256 CBC

The system uses the AES algorithm in Cipher Block Chaining (CBC) mode. In general one avoids using encryption 'as-is' (i.e. Electronic Code Book), since it means any piece of plaintext always maps to the same piece of ciphertext. Cipher Block Chaining ensures that if you encrypted the same thing a bunch of times it would always encrypt to a new piece of ciphertext.

You can see another reference on the design of the encryption side; we'll be only talking about the decryption side here. In this case AES-256 CBC mode is used as follows, where the details of the AES-256 Decryption block will be discussed in detail later:

![AES-256](https://wiki.newae.com/images/8/88/Aes256_cbc.png)

This diagram shows that the output of the decryption is no longer used directly as the plaintext. Instead, the output is XORed with a 16 byte mask, which is usually taken from the previous ciphertext. Also, the first decryption block has no previous ciphertext to use, so a secret initialization vector (IV) is used instead. If we are going to decrypt the entire ciphertext (including block 0) or correctly generate our own ciphertext, we'll need to find this IV along with the AES key.

### Attacking AES-256

The system in this tutorial uses AES-256 encryption, which has a 256 bit (32 byte) key - twice as large as the 16 byte key we've attacked in previous tutorials. This means that our regular AES-128 CPA attacks won't quite work. However, extending these attacks to AES-256 is fairly straightforward: the theory is explained in detail in [Extending AES-128 Attacks to AES-256](Extending%20AES-128%20Attacks%20to%20AES-256.ipynb).

As the theory page explains, our AES-256 attack will have 4 steps:

1. Perform a standard attack (as in AES-128 decryption) to determine the first 16 bytes of the key, corresponding to the 14th round encryption key.
1. Using the known 14th round key, calculate the hypothetical outputs of each S-Box from the 13th round using the ciphertext processed by the 14th round, and determine the 16 bytes of the 13th round key manipulated by inverse MixColumns.
1. Perform the MixColumns and ShiftRows operation on the hypothetical key determined above, recovering the 13th round key.
1. Using the AES-256 key schedule, reverse the 13th and 14th round keys to determine the original AES-256 encryption key.

## Firmware

For this tutorial, we'll be using the `bootloader-aes256` project, which we'll build as usual:

In [ ]:
%%bash -s "$PLATFORM" 
cd ../../../hardware/victims/firmware/bootloader-aes256
make PLATFORM=$1 CRYPTO_TARGET=NONE

## Capturing Traces

### Setup

To start, we'll proceed with setup as usual:

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../../hardware/victims/firmware/bootloader-aes256/bootloader-aes256-{}.hex".format(PLATFORM)

In [ ]:
cw.program_target(scope, prog, fw_path)

### Calculating the CRC

The next step we'll need to take in attacking this target is to communicate with it. Most of the transmission is fairly straight forward, but the CRC is a little tricky. Luckily, there's a lot of open source out there for calculating CRCs. In this case, we'll pull some code from pycrc:

In [ ]:
# Class Crc
#############################################################
# These CRC routines are copy-pasted from pycrc, which are:
# Copyright (c) 2006-2013 Thomas Pircher <tehpeh@gmx.net>
#
class Crc(object):
    """
    A base class for CRC routines.
    """

    def __init__(self, width, poly):
        """The Crc constructor.

        The parameters are as follows:
            width
            poly
            reflect_in
            xor_in
            reflect_out
            xor_out
        """
        self.Width = width
        self.Poly = poly


        self.MSB_Mask = 0x1 << (self.Width - 1)
        self.Mask = ((self.MSB_Mask - 1) << 1) | 1

        self.XorIn = 0x0000
        self.XorOut = 0x0000

        self.DirectInit = self.XorIn
        self.NonDirectInit = self.__get_nondirect_init(self.XorIn)
        if self.Width < 8:
            self.CrcShift = 8 - self.Width
        else:
            self.CrcShift = 0

    def __get_nondirect_init(self, init):
        """
        return the non-direct init if the direct algorithm has been selected.
        """
        crc = init
        for i in range(self.Width):
            bit = crc & 0x01
            if bit:
                crc ^= self.Poly
            crc >>= 1
            if bit:
                crc |= self.MSB_Mask
        return crc & self.Mask


    def bit_by_bit(self, in_data):
        """
        Classic simple and slow CRC implementation.  This function iterates bit
        by bit over the augmented input message and returns the calculated CRC
        value at the end.
        """
        # If the input data is a string, convert to bytes.
        if isinstance(in_data, str):
            in_data = [ord(c) for c in in_data]

        register = self.NonDirectInit
        for octet in in_data:
            for i in range(8):
                topbit = register & self.MSB_Mask
                register = ((register << 1) & self.Mask) | ((octet >> (7 - i)) & 0x01)
                if topbit:
                    register ^= self.Poly

        for i in range(self.Width):
            topbit = register & self.MSB_Mask
            register = ((register << 1) & self.Mask)
            if topbit:
                register ^= self.Poly

        return register ^ self.XorOut
    
bl_crc = Crc(width = 16, poly=0x1021)

Now we can easily get the CRC for our message by calling `bl_crc.bit_by_bit(message)`. 

### Communicating with the Bootloader

With that done, we can start communicating with the bootloader. The following block will ensure that the target has finished booting:

In [ ]:
import time
okay = 0
reset_target(scope)

while not okay:
    target.write('\0xxxxxxxxxxxxxxxxxx')
    time.sleep(0.05)
    response = target.read()
    if response:
        print(response)
        if ord(response[0]) == 0xA1:
            okay = 1

Recall that the bootloader expects:

* To start with `0x00`
* A 16 byte encrypted message (4 bytes signature + 12 bytes data)
* CRC16

We don't really care what the 16 byte message is (just that each is different so that we get a variety of hamming weights), so we'll use the same text/key module from earlier attacks.

We can now run the following block, and we should get `0xA4` back. You may need to run this block a few times to get the right response back.

In [ ]:
import time
message = [0x00]
ktp = cw.ktp.Basic()

# clear serial buffer
print(target.read())

key, text = ktp.next() #don't care about key here
message.extend(text)

crc = bl_crc.bit_by_bit(text)

message.append(crc >> 8)
message.append(crc & 0xFF)

target.write(message)
time.sleep(0.1)

response = target.read()
print("Response: {:02X}".format(ord(response[0])))

### Capturing Traces

With that out of the way, we can proceed to capturing our traces. The normal 5000 traces we capture isn't long enough to get the rounds we care about, so we'll need to increase it (15000 should be fine):

In [ ]:
scope.adc.samples = 15000

We'll be working with Analyzer, so we'll need to use a ChipWhisperer project to store our traces and text:

In [ ]:
project = cw.create_project("projects/Tutorial_A5", overwrite=True)
ktp = cw.ktp.Basic()

Below you'll find our capture loop. This will be pretty similar to Tutorial B5, but we've added our communication code. We also check the response and just skip the data we get if it isn't correct.

In [ ]:
#Capture Traces
from tqdm.notebook import trange
import numpy as np
import time
N = 200  # Number of traces
for i in trange(N, desc='Capturing traces'):
    message = [0x00]
    target.read()
    
    key, text = ktp.next()
    message.extend(text)
    
    crc = bl_crc.bit_by_bit(text)
    message.append(crc >> 8)
    message.append(crc & 0xFF)

    scope.arm()

    target.write(message)
    
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')
    response = target.read()
    if ord(response[0]) != 0xA4:
        # Bad response, just skip
        print("Bad response: {:02X}".format(ord(response[0])))
        continue
    
    project.traces.append(cw.Trace(scope.get_last_trace(), text, "", key))

## Analysis

Now that we have our traces, we can go ahead and perform the attack. As described in the background theory, we'll have to do two attacks - one to get the 14th round key, and another (using the first result) to get the 13th round key. Then, we'll do some post-processing to finally get the 256 bit encryption key.

### 14th Round Key

We can attack the 14th round key with a standard, no-frills CPA attack (using the inverse sbox, since it's a decryption that we're breaking):

In [ ]:
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa

leak_model = cwa.leakage_models.inverse_sbox_output

attack = cwa.cpa(project, leak_model)

With the setup done, we can actually preform the attack. 11000 samples is a rather large amount to chew through, so if you want a faster attack you can use a smaller range in `attack.point_range`. `(2900, 4200)` will work for XMEGA, while `(1400, 2600)` will work for the STM32F3 (CWLite ARM).

In [ ]:
key = [0xea, 0x79, 0x79, 0x20, 0xc8, 0x71, 0x44, 0x7d, 0x46, 0x62, 0x5f, 0x51, 0x85, 0xc1, 0x3b, 0xcb]

cb = cwa.get_jupyter_callback(attack)
if PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    attack.point_range = [1400, 2600]
elif PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    pass
attack_results = attack.run(cb)

In [ ]:
rec_key = []
for bnum in attack_results.find_maximums():
    rec_key.append(bnum[0][0])
    print("Best Guess = 0x{:02X}, Corr = {}".format(bnum[0][0], bnum[0][2]))

### 13th Round Key

Analyzer doesn't have a leakage model for the 13th round key built in, so we'll need to create our own. Let's work through this now. To make a new model, we start off by inheriting the `AESLeakageHelper` class. We need to make a `leakage()` method that calculates the Hamming weight we use in the CPA attack. To get you started:

In [ ]:
class AES256_Round13_Model(cwa.AESLeakageHelper):
    def leakage(self, pt, ct, guess, bnum):
        #You must put YOUR recovered 14th round key here - this example may not be accurate!
        calc_round_key = [0xea, 0x79, 0x79, 0x20, 0xc8, 0x71, 0x44, 0x7d, 0x46, 0x62, 0x5f, 0x51, 0x85, 0xc1, 0x3b, 0xcb]
        state = reverse_round_14(self, pt, calc_round_key)
        state = reverse_round_13(self, state) #reverse state just before inv_subbytes
        return self.inv_sbox(state[bnum] ^ guess[bnum])

Now we just need to make the `reverse_round_14()` and `reverse_round_13()` functions. By passing the class in, we get access to `self.inv_shiftrows()`, `self.inv_subbytes()`, and `self.inv_mixcolumns()`.

In [ ]:
def reverse_round_14(self, pt, key):
    state = [pt[i] ^ key[i] for i in range(16)] #AddRoundKey
    state = self.inv_shiftrows(state)
    state = self.inv_subbytes(state)
    return state # we're now at the end of decryption round 1

def reverse_round_13(self, state):
    state = self.inv_mixcolumns(state)
    state = self.inv_shiftrows(state)
    return state

We can now make a new leakage model, and set our attack to use it instead of the other one:

In [ ]:
leak_model = cwa.leakage_models.new_model(AES256_Round13_Model)
attack.leak_model = leak_model

#### Resyncing Traces (XMEGA Only)

The traces for the XMEGA version of the firmware become desynced around sample 7000. This is due to a non-constant AES implementation: the code does not always take the same amount of time to run for every input. (It's actually possible to do a timing attack on this AES implementation! We'll stick with our CPA attack for now.)

While this does open up a timing attack, it actually makes our AES attack a little harder, since we'll have to resync the traces. Luckily, this can be done pretty easily by using the ResyncSAD preprocessing module:

In [ ]:
if PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    resync_traces = cwa.preprocessing.ResyncSAD(project)
    resync_traces.enabled = True
    resync_traces.ref_trace = 0
    resync_traces.target_window = (9100, 9300)
    resync_traces.max_shift = 200
    attack.change_project(resync_traces.preprocess())

#### Running the Attack

Like in the 14th round attack, we can use a smaller range of points to make the attack faster. `(8000,10990)` works well for the XMEGA, while `(6500, 8500)` works well for the STM32F3.

In [ ]:
if PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    attack.point_range = [6500,8500]
elif PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    attack.point_range = [8000,10990]
cb = cwa.get_jupyter_callback(attack)
attack_results = attack.run(cb)

You can run the block below and the correct key should be printed out:

In [ ]:
rec_key2 = []
for bnum in attack_results.find_maximums():
    print("Best Guess = 0x{:02X}, Corr = {}".format(bnum[0][0], bnum[0][2]))
    rec_key2.append(bnum[0][0])

This, however, isn't actually the 13th round key. To get the real 13th round key, we'll need to run what we've recovered through a `shiftrows()` and `mixcolumns()` operation:

In [ ]:
real_key2 = cwa.aes_funcs.shiftrows(rec_key2)
real_key2 = cwa.aes_funcs.mixcolumns(real_key2)

print("Recovered:", end="")
for subkey in real_key2:
    print(" {:02X}".format(subkey), end="")
print("")

We now have everything we need to recover the full key! We'll start by combining the 13th and 14th round keys:

In [ ]:
rec_key_comb = real_key2.copy()
rec_key_comb.extend(rec_key)

print("Key:", end="")
for subkey in rec_key_comb:
    print(" {:02X}".format(subkey), end="")
print("")

and then we can use the `AES128_8bit` leakage model to recover the first two rounds:

In [ ]:
btldr_key = leak_model.key_schedule_rounds(rec_key_comb, 13, 0)
btldr_key.extend(leak_model.key_schedule_rounds(rec_key_comb, 13, 1))
print("Key:", end="")
for subkey in btldr_key:
    print(" {:02X}".format(subkey), end="")
print("")

You should see a 32 byte key printed out. Open `supersecret.h`, confirm that we have the right key, and celebrate! 

## Recovering the IV

Now that we have the encryption key, we can proceed onto an attack of the next secret value: the IV.

Here, we have the luxury of seeing the source code of the bootloader. This is generally not something we would have access to in the real world, so we'll try not to use it to cheat. (peeking at `supersecret.h` counts as cheating). Instead, we'll use the source to help us identify important parts of the power traces.

### Bootloader Source Code

Inside the bootloader's main loop, it does three tasks that we're interested in:

* it decrypts the incoming ciphertext;
* it applies the IV to the decryption's result; and
* it checks for the signature in the resulting plaintext.

This snippet from `bootloader.c` shows all three of the tasks:

```C
// Continue with decryption
trigger_high();                
aes256_decrypt_ecb(&ctx, tmp32);
trigger_low();
             
// Apply IV (first 16 bytes)
for (i = 0; i < 16; i++){
    tmp32[i] ^= iv[i];
}

//Save IV for next time from original ciphertext                
for (i = 0; i < 16; i++){
    iv[i] = tmp32[i+16];
}

// Tell the user that the CRC check was okay
putch(COMM_OK);
putch(COMM_OK);

//Check the signature
if ((tmp32[0] == SIGNATURE1) &&
   (tmp32[1] == SIGNATURE2) &&
   (tmp32[2] == SIGNATURE3) &&
   (tmp32[3] == SIGNATURE4)){
   
   // Delay to emulate a write to flash memory
   _delay_ms(1);
}   
```

This gives us a pretty good idea of how the microcontroller is going to do its job, but if you'd like to go further, you can open the `.lss` file for the binary that was built. This is called a listing file and it lets you see the assembly that the C was compiled and linked to.

### Power Traces

As you can see from both files, after the decryption process, the bootloader executes a few distinct pieces of code:

* To apply the IV, it uses an XOR operation;
* To store the new IV, it copies the previous ciphertext into the IV array;
* It sends two bytes on the serial port;
* It checks the bytes of the signature one by one.

We should be able to recognize these four parts of the code in the power traces. Let's modify our capture routine to find them:

1. We're looking for the original IV, but it's overwritten after each successful decryption. This means we'll have to reset the target before each trace we capture
1. We'd like to skip over all of the decryption process. Recall that the trigger pin is set low after the decryption finishes. This means we can skip over the AES-256 function by triggering on a falling edge instead
1. Depending on the target, we may have to flush the target's serial lines by sending it a bunch of invalid data and looking for a bad CRC return. This slows down the capture process by a lot, so you may want to try without doing this first.
1. We won't need as many samples, so we can reduce how many we capture. 3000 should be sufficient for most targets.

Let's start by reducing our samples and making a function to reset our target (depending on your target, you may need to change the reset pin):

In [ ]:
import time
scope.adc.samples = 3000

We can trigger on a falling edge by changing `scope.adc.basic_mode` to `"falling_edge"`:

In [ ]:
scope.adc.basic_mode = "falling_edge"

We can flush the serial line by sending an invalid message, then checking for a bad CRC return value (`0xA1`). Let's make sure our changes work by getting a trace:

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
reset_target(scope)
message = [0x00]

target.read()

key, text = ktp.new_pair()  # manual creation of a key, text pair can be substituted here

message.extend(text)

crc = bl_crc.bit_by_bit(text)
message.append(crc >> 8)
message.append(crc & 0xFF)

scope.arm()

okay = 0
while not okay:
    target.write(message)
    time.sleep(0.005)
    response = target.read()
    if response:
        if ord(response[0]) == 0xA4:
            okay = 1
ret = scope.capture()

trace = scope.get_last_trace()

output_notebook()
p = figure()

xrange = range(len(trace))
p.line(xrange, trace, line_color="red")
show(p)

You should see 5 different sections:

* 16 XORs
* 16 register loads (this is the new IV being copied over)
* Some serial communication
* The signature check
* The serial line going idle

Different targets have different power traces (for example, on Arm the XORs and register loads are almost identical), but hopefully you can pick out where each section is. For example, on XMEGA:

![XMEGA_Bonus_Trace](https://wiki.newae.com/images/f/f6/Tutorial-A5-Bonus-Trace-Notes.PNG)

With all of these things clearly visible, we have a pretty good idea of how to attack the IV and the signature. We should be able to look at each of the XOR spikes to find each of the IV bytes - each byte is processed on its own. Then, the signature check uses a short-circuiting comparison: as soon as it finds a byte in error, it stops checking the remaining bytes. This type of check is susceptible to a timing attack.

With those things done, we can move onto our capture loop. It's pretty similar to our last one. We're done with Analyzer, so we can store our traces in Python lists (we'll convert to numpy arrays later for easy analysis).

In [ ]:
from tqdm.notebook import trange
import numpy as np
import time
traces = []
keys = []
plaintexts = []

if PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
    N = 500  # Number of traces
elif PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
    N=5000 #oof DPA attacks on XMEGA
for i in trange(N, desc='Capturing traces'):
    reset_target(scope)
    message = [0x00]
    
    target.read()
    
    key, text = ktp.new_pair()  # manual creation of a key, text pair can be substituted here
    keys.append(key)
    plaintexts.append(text)
    
    message.extend(text)
    
    crc = bl_crc.bit_by_bit(text)
    message.append(crc >> 8)
    message.append(crc & 0xFF)
    scope.arm()
    
    
    okay = 0
    passes = 0
    while not okay:
        target.write(message)
        time.sleep(0.01)
        response = target.read()
        passes += 1
        if response:
            if ord(response[0]) == 0xA4:
                okay = 1
            elif passes >= 100:
                break
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')
        continue
    
    traces.append(scope.get_last_trace())

### Analysis

#### Attack Theory

The bootloader applies the IV to the AES decryption result (`DR`) by calculating


$\text{PT} = \text{DR} \oplus \text{IV}$

where DR is the decrypted ciphertext, IV is the secret vector, and PT is the plaintext that the bootloader will use later. We only have access to one of these: since we know the AES-256 key, we can calculate DR. This exclusive or will be visible in the power traces.

This is enough information for us to attack a single bit of the IV. Suppose we only wanted to get the first bit (bit 0) of the first byte (byte 0) of the IV. We could do the following:

* Split all of the traces into two groups: those with `(DR[0] & 0x01) = 0`, and those with `(DR[0] & 0x01) = 1`. 
* Calculate the average trace for both groups.
* Find the difference between the two averages. Provided we've got sufficient data, we should see a spike where the xor is occuring.
* Look at the direction of the spike to decide if the IV bit is 0 `(PT[0] = DR[0])` or if the IV bit is 1 `(PT[0] = ~DR[0])`.

This is effectively a DPA attack on a single bit of the IV. We can repeat this attack across the whole IV by instead separating by `(DR[byte] & (1 << bit) = 0` and `(DR[byte] & (1 << bit) = bit`.

#### Finding the XOR

Recall that we're looking for the xor operation between the last decrypted block (`DR`), so we'll need to decrypt it up to that point. PyCryptoDome includes an AES decryption routine, so we'll be using that. We'll start by importing the necessary modules and converting our traces/plaintext to numpy arrays:

In [ ]:
from Crypto.Cipher import AES
import numpy as np

trace_array = np.asarray(traces)  # if you prefer to work with numpy array for number crunching
textin_array = np.asarray(plaintexts)

numTraces = len(trace_array)
traceLen = len(trace_array[0])

Next we'll do the AES256 decryption. If you got a different key in the earlier part, you'll need to change `knownkey`.

In [ ]:
knownkey = [0x94, 0x28, 0x5D, 0x4D, 0x6D, 0xCF, 0xEC, 0x08, 0xD8, 0xAC, 0xDD, 0xF6, 0xBE, 0x25, 0xA4, 0x99,
            0xC4, 0xD9, 0xD0, 0x1E, 0xC3, 0x40, 0x7E, 0xD7, 0xD5, 0x28, 0xD4, 0x09, 0xE9, 0xF0, 0x88, 0xA1]

knownkey = bytes(knownkey)
dr = []
aes = AES.new(knownkey, AES.MODE_ECB)
for i in range(numTraces):
    ct = bytes(textin_array[i])
    pt = aes.decrypt(ct)
    d = [bytearray(pt)[i] for i in range(16)]
    dr.append(d)

Now that we have `DR`, we can move on to the next step of our attack plan by separating our traces. We'll do this for each bit of the first byte, since it'll make it easier to identify the xor operation when the time comes:

In [ ]:
grouped_byte_traces = []
byte = 0

for bit in range(8):
    grouped_bit_traces = [], []
    for i in range(numTraces):
        if (dr[i][byte] & (1 << bit)):
            grouped_bit_traces[0].append(trace_array[i])
        else:
            grouped_bit_traces[1].append(trace_array[i])
    grouped_byte_traces.append(grouped_bit_traces)
    print(len(grouped_bit_traces[0]))

If you have 1000 traces, you should expect this to print a number around 500 - roughly half of the traces should fit into each group. Now, NumPy's average function lets us easily calculate the average at each point:

In [ ]:
# Find averages and differences
diffs = []
for i in range(8):
    means = np.average(grouped_byte_traces[i][0], axis=0), np.average(grouped_byte_traces[i][1], axis=0)
    diffs.append(means[1] - means[0])

Finally, we'll need to locate the XOR operation for the first IV byte. Since each bit is being XORed in the same operation, each should have a spike in the same spot. Therefore, if we plot all the bits, there should be a single spot where every bit has a spike:

In [ ]:
# Split traces into 2 groups
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

output_notebook()
p = figure()

xrange = range(len(diffs[0]))
xrange2 = range(len(traces[0]))
colours = ["red", "blue", "green", "black"]
for i in range(8):
    p.line(xrange, diffs[i], line_color=colours[i%4])
show(p)

If you zoom into the beginning of the plot, you should be able to locate a few potential locations for the XOR. We could manually look through the plot to find potential XOR locations, but we can make our lives a little easier by selecting potential locations by each diff having a spike in that location. We'll select a location based on the value of the largest spike divided by 3. This isn't a rigourous selection: if you get an overwhelming number of points, try increasing the threshold. If you don't get any, reduce the threshold.

In [ ]:
def find_potential_xors(diffs):
    threshold = np.max(abs(np.array(diffs))) / 3
    print(threshold)
    interesting_pts = []
    for pt in range(len(diffs[0][:1400])):
        interesting = True
        for bit in range(8):
            if abs(diffs[bit][pt]) < threshold:
                interesting = False
        if interesting:
            interesting_pts.append(pt)
    return interesting_pts

In [ ]:
find_potential_xors(diffs)

We're almost there! To narrow the selection further, we can find interesting points of a few other bytes as well. Since the same operation is being performed on each byte, there should be a constant offset between the spikes of different bytes. Let's repeat the process on a few different bytes and see what we get:

In [ ]:
def get_diffs(byte):
    grouped_byte_traces = []

    for bit in range(8):
        grouped_bit_traces = [], []
        for i in range(numTraces):
            if (dr[i][byte] & (1 << bit)):
                grouped_bit_traces[0].append(trace_array[i])
            else:
                grouped_bit_traces[1].append(trace_array[i])
        grouped_byte_traces.append(grouped_bit_traces)
        
    # Find averages and differences
    diffs = []
    for i in range(8):
        means = np.average(grouped_byte_traces[i][0], axis=0), np.average(grouped_byte_traces[i][1], axis=0)
        diffs.append(means[1] - means[0])
    return diffs

for i in range(5):
    diffs = get_diffs(i)
    print(find_potential_xors(diffs))

Again, you might have to adjust the threshold based on how many values you get. Try to get a few bytes that only have a few potential values.

Now that you have some peak data, you'll want to use this to find the time shift between XORs. This time shift should be constant between samples and needs to work for all samples (each run through the loop is the same, so it makes sense that the time shift should be constant). We'd also expect that the difference between the peaks would be a multiple of whole clock cycles. We're sampling at x4 the devices clock, so the offset should be divisible by 4. For example, you might have:

```
0th byte @ 41, 42
1st byte @ 77, 81
2nd byte @ 117, 121
3rd byte @ 157, 158, 159, 161, 162
4th byte @ 197, 198, 201, 202
```

Starting off, we have 41 and 42 as potential locations. The next byte only has odd locations, so our first XOR likely occured at sample 41. 41 could lead to either 77 (offset of 36) or 81 (offset of 40). 77 doesn't lead to any interesting points in byte 2 with an offset of 36, but 81 does lead onto 121, which will lead onto 161, then onto 201. Therefore, our offset is likely 40.

As an exercise, try writing some python code to automatically find the starting location and offset.

In [ ]:
#your code here...

#### The Other 127

The best way to attack the IV would be to repeat the 1-bit conceptual attack for each of the bits. Try to do this yourself! (Really!) If you're stuck, here are a few hints to get you going:

One easy way of looping through the bits is by using two nested loops, like this:

```python
for byte in range(16):
    for bit in range(8):
        # Attack bit number (byte*8 + bit)
```

The sample that you'll want to look at will depend on which byte you're attacking. We had success when we used `location = 51 + byte*60`, but your mileage will vary.

The bitshift operator and the bitwise-AND operator are useful for getting at a single bit:

```python
# This will either result in a 0 or a 1
checkIfBitSet = (byteToCheck >> bit) & 0x01
```

In [ ]:
btldr_IV = [0] * 16 #181
for byte in range(16):
    if PLATFORM == "CWLITEARM" or PLATFORM == "CW308_STM32F3":
        location = 53 + byte * 36
    elif PLATFORM == "CWLITEXMEGA" or PLATFORM == "CW303":
        location = 55 + byte * 60
    iv = 0
    for bit in range(8):
        pt_bits = [((dr[i][byte] >> (7-bit)) & 0x01) for i in range(numTraces)]

        # Split traces into 2 groups
        groupedPoints = [[] for _ in range(2)]
        for i in range(numTraces):
            groupedPoints[pt_bits[i]].append(trace_array[i][location])
            
        means = []
        for i in range(2):
            means.append(np.average(groupedPoints[i]))
        diff = means[1] - means[0]
        
        iv_bit = 1 if diff > 0 else 0
        iv = (iv << 1) | iv_bit
        
        print(iv_bit, end = " ")
        
    print("{:02X}".format(iv))
    btldr_IV[byte] = iv
    
print(btldr_IV)

## Attacking the Signature

The last thing we can do with this bootloader is attack the signature. This final section will show how one byte of the signature could be recovered. If you want more of this kind of analysis, a more complete timing attack is shown in Tutorial B3-1 Timing Analysis with Power for Password Bypass.

### Attack Theory

Recall from earlier that the signature check in C looks like:

```C
if ((tmp32[0] == SIGNATURE1) &&
    (tmp32[1] == SIGNATURE2) &&
    (tmp32[2] == SIGNATURE3) &&
    (tmp32[3] == SIGNATURE4)){
```

In C, boolean expressions support short-circuiting. When checking multiple conditions, the program will stop evaluating these booleans as soon as it can tell what the final value will be. In this case, unless all four of the equality checks are true, the result will be false. Thus, as soon as the program finds a single false condition, it's done.

Open the listing file for your binary (`.lss`), find the signature check, and confirm that this is happening. For example, on the STM32F3, the assembly looks like this:

```
                //Check the signature
                if ((tmp32[0] == SIGNATURE1) &&
 8000338:	f89d 3018 	ldrb.w	r3, [sp, #24]
 800033c:	2b00      	cmp	r3, #0
 800033e:	d1c2      	bne.n	80002c6 <main+0x52>
 8000340:	f89d 2019 	ldrb.w	r2, [sp, #25]
 8000344:	2aeb      	cmp	r2, #235	; 0xeb
 8000346:	d1be      	bne.n	80002c6 <main+0x52>
                   (tmp32[1] == SIGNATURE2) &&
 8000348:	f89d 201a 	ldrb.w	r2, [sp, #26]
 800034c:	2a02      	cmp	r2, #2
 800034e:	d1ba      	bne.n	80002c6 <main+0x52>
                   (tmp32[2] == SIGNATURE3) &&
 8000350:	f89d 201b 	ldrb.w	r2, [sp, #27]
 8000354:	2a1d      	cmp	r2, #29
 8000356:	d1b6      	bne.n	80002c6 <main+0x52>
                   (tmp32[3] == SIGNATURE4)){
```

This assembly code confirms the short-circuiting operation. Each of the four assembly blocks include a comparison and a conditional branch. All four of the conditional branches (`bne.n`) return the program to the same location (the start of the `while(1)` loop). All four branches must fail to get into the body of the if block.

The short-circuiting conditions are perfect for us. We can use our power traces to watch how long it takes for the signature check to fail. If the check takes longer than usual, then we know that the first byte of our signature was right.

### Power Traces

Our capture loop will be pretty similar to the one we used to break the IV, but now that we know the secret values of the encryption process we can make some improvements by encrypting the text that we send. This has two important advantages:

1. We can control the signature. We could reuse the traces we took during the IV attack, but this way ensures that we hit each possible value once. It also simplifies the analysis, since we don't have to worry about decrypting the text we sent.
1. We no longer have to reset after each attempt, since we know what the next IV is going to be (we do need to reset at the beginning to make sure we're on the same starting IV as the target). This speeds up the capture process considerably. 

To perform the AES256 CBC encryption, there's a few steps we need to take:

1. XOR the IV with the text we want to send
1. Encrypt this new text
1. Set this cipher text as the new IV

We can use PyCrypto again to make the encryption process easy and the other two steps are simple operations. We'll run our loop 256 times (one for each possible byte value) and assign that value to the byte we want to check. We're not quite sure where the check is happening, so we'll be safe and capture 24000 traces. Everthing else should look familiar from earlier parts of the tutorial:

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
from Crypto.Cipher import AES
import time

traces = []
keys = []
plaintexts = []

iv = [0xC1, 0x25, 0x68, 0xDF, 0xE7, 0xD3, 0x19, 0xDA, 0x10, 0xE2, 0x41, 0x71, 0x33, 0xB0, 0xEB, 0x3C]

knownkey = [0x94, 0x28, 0x5D, 0x4D, 0x6D, 0xCF, 0xEC, 0x08, 0xD8, 0xAC, 0xDD, 0xF6, 0xBE, 0x25, 0xA4, 0x99,
            0xC4, 0xD9, 0xD0, 0x1E, 0xC3, 0x40, 0x7E, 0xD7, 0xD5, 0x28, 0xD4, 0x09, 0xE9, 0xF0, 0x88, 0xA1]

knownkey = bytes(knownkey)
aes = AES.new(knownkey, AES.MODE_ECB)
N = 256 # Number of traces

reset_target(scope)
okay=0
scope.adc.basic_mode = "falling_edge"
while not okay:
    target.write("\0xxxxxxxxxxxxxxxxxx")
    time.sleep(0.005)
    response = target.read()
    if response:
        if ord(response[0]) == 0xA1:
            okay = 1

scope.adc.samples = 24000
scope.adc.offset = 0
for byte in trange(N, desc='Attacking Signature Byte'):
    message = [0x00]
    text = [0] * 16
    
    # the 4 signature bytes
    text[0] = byte
    text[1] = 0
    text[2] = 0
    text[3] = 0
    
    target.read()
    
    textcpy = [0] * 16
    textcpy[:] = text[:]
    plaintexts.append(textcpy)
    
    # Apply IV
    for i in range(len(iv)):
        text[i] ^= iv[i]
    
    # Encrypt text
    ct = aes.encrypt(bytes(text))
    
    message.extend(ct)
    
    # Use ct as new IV
    iv[:] = ct[:]
    
    crc = bl_crc.bit_by_bit(ct)
    message.append(crc >> 8)
    message.append(crc & 0xFF)
    
    scope.arm()

    target.write(message)
    timeout = 50
    
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')
        continue
        
    response = target.read()
    if ord(response[0]) != 0xA4:
        # Bad response, just skip
        print("Bad response: {:02X}".format(ord(response[0])))
        continue
    
    traces.append(scope.get_last_trace())

### Analysis

Now that we've captured our traces, the actual analysis is pretty simple. We're looking for a single trace that looks very different from the rest. A simple way to find this is to compare all the traces to a reference trace. We'll use the average of all the traces as our reference:

In [ ]:
mean = np.average(traces, axis=0)

That leaves us with comparing the traces. Let's start by plotting the difference between some of the traces and the mean:

In [ ]:
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

output_notebook()
p = figure()
colors = ["red", "blue", "green", "yellow"]
for i in range(0,10):
    p.line(range(len(traces[i])), traces[i]-mean, line_color=colors[i%4])
        
show(p)

Depending on your target, you might have seen something like this:

![](https://wiki.newae.com/images/2/25/Bokeh_plot_%285%29.png)

Looks like we've found our trace! However, let's clean this up with some statistics. We can use the correlation coefficient to see which bytes are the furthest away from the average. We only want to take the correlation across where the plots differ, chose a subset of the plot where there's a large difference. In the case of the above picture, the difference starts at around 18k, and continues until the end. A range of 18000 to 20000 should work nicely:

In [ ]:
corr = []
for i in range(256):
    corr.append(np.corrcoef(mean[18000:20000], traces[i][18000:20000])[0, 1])
print(np.sort(corr))
print(np.argsort(corr))

This output tells us two things:

* The first list says that almost every trace looks very similar to the overall mean (98% correlated or higher). However, there's one trace that is totally different with much lower correlation. This is probably our correct guess.
* The second list gives the signature guess that matches each of the above correlations. The first number in the list is 0x00, which is the correct signature!

To finish this attack, change the capture loop to keep the first byte fixed and vary the second byte instead. Repeat this with the rest of the bytes and you should have the signature.

In [ ]:
from tqdm.notebook import tqdm
import numpy as np
from Crypto.Cipher import AES
import time

traces = []
keys = []
plaintexts = []
btldr_sig = [0] * 4

iv = [0xC1, 0x25, 0x68, 0xDF, 0xE7, 0xD3, 0x19, 0xDA, 0x10, 0xE2, 0x41, 0x71, 0x33, 0xB0, 0xEB, 0x3C]

knownkey = [0x94, 0x28, 0x5D, 0x4D, 0x6D, 0xCF, 0xEC, 0x08, 0xD8, 0xAC, 0xDD, 0xF6, 0xBE, 0x25, 0xA4, 0x99,
            0xC4, 0xD9, 0xD0, 0x1E, 0xC3, 0x40, 0x7E, 0xD7, 0xD5, 0x28, 0xD4, 0x09, 0xE9, 0xF0, 0x88, 0xA1]

knownkey = bytes(knownkey)
aes = AES.new(knownkey, AES.MODE_ECB)
N = 256 # Number of traces

reset_target(scope)
okay=0
scope.adc.basic_mode = "falling_edge"
while not okay:
    target.write("\0xxxxxxxxxxxxxxxxxx")
    time.sleep(0.005)
    response = target.read()
    if response:
        if ord(response[0]) == 0xA1:
            okay = 1
            
scope.adc.samples = 24000
scope.adc.offset = 0
for bnum in range(4):
    traces = []
    for byte in trange(N, desc='Attacking Signature Byte {}'.format(bnum)):
        message = [0x00]
        text = [0] * 16

        # the 4 signature bytes
        for j in range(bnum):
            text[j] = btldr_sig[j]
        text[bnum] = byte
        
        target.read()

        textcpy = [0] * 16
        textcpy[:] = text[:]
        plaintexts.append(textcpy)

        # Apply IV
        for i in range(len(iv)):
            text[i] ^= iv[i]

        # Encrypt text
        ct = aes.encrypt(bytes(text))

        message.extend(ct)

        # Use ct as new IV
        iv[:] = ct[:]

        crc = bl_crc.bit_by_bit(ct)
        message.append(crc >> 8)
        message.append(crc & 0xFF)

        scope.arm()
        target.write(message)
        ret = scope.capture()
        if ret:
            print('Timeout happened during acquisition')
            continue

        # run aux stuff that should happen after trace here
        response = target.read()
        if ord(response[0]) != 0xA4:
            # Bad response, just skip
            print("Bad response: {:02X}".format(ord(response[0])))
            continue

        traces.append(scope.get_last_trace())
        
    mean = np.average(traces, axis=0)
    corr = []
    for i in range(256):
        corr.append(np.corrcoef(mean[18000:20000], traces[i][18000:20000])[0, 1])
    btldr_sig[bnum] = np.argsort(corr)[0]
    
print(btldr_sig)

In [ ]:
scope.dis()
target.dis()

## Conclusion

We've now successfully recovered all of the secrets of the bootloader!

## Tests

In [ ]:
real_btldr_key = [0x94, 0x28, 0x5D, 0x4D, 0x6D, 0xCF, 0xEC, 0x08, 0xD8, 0xAC, 0xDD, 0xF6, 0xBE, 0x25, 0xA4, 0x99, \
                    0xC4, 0xD9, 0xD0, 0x1E, 0xC3, 0x40, 0x7E, 0xD7, 0xD5, 0x28, 0xD4, 0x09, 0xE9, 0xF0, 0x88, 0xA1]

real_btldr_IV = [0xC1, 0x25, 0x68, 0xDF, 0xE7, 0xD3, 0x19, 0xDA, 0x10, 0xE2, 0x41, 0x71, 0x33, 0xB0, 0xEB, 0x3C]

real_btldr_sig = [0x00, 0xEB, 0x02, 0x1D]

In [ ]:
assert (btldr_key == list(real_btldr_key)), "Attack on encryption key failed!\nGot: {}\nExp: {}".format(btldr_key, real_btldr_key)

In [ ]:
assert (btldr_IV == real_btldr_IV), "Attack on IV failed!\nGot: {}\nExpected: {}".format(btldr_IV, real_btldr_IV)

In [ ]:
assert (btldr_sig == real_btldr_sig), "Attack on signature failed!\nGot: {}\nExpected: {}".format(btldr_sig, real_btldr_sig)

## Task 2

**SUMMARY:** *Through the previous labs, we've gained a lot of tools to attack unknown embedded devices: SPA, DPA, CPA, trace resynchronization, and more. In this lab, we'll be using some of those techniques to break a more realistic target: a bootloader. Note that there are two versions of this lab. In this one (Lab B), we'll start with no information that couldn't be revealed by watching code be sent to the bootloader. Everything else, we'll need to figure out for ourselves, such as what encryption algorithm the target is using, how it's using it, etc. In lab A, this information will be given and you'll just focus on the attack. It's up to you whether you want to run this lab or Lab A!*

**LEARNING OUTCOMES:**

* Observing power traces to figure out what encryption operation it's running
* Applying CPA and DPA to break different parts of the bootloader
* Understanding different operating modes for block ciphers

## Background

In the world of microcontrollers, a bootloader is a special piece of firmware that is made to let the user upload new programs into memory. This is especially useful for devices with complex code that may need to be patched or otherwise updated in the future - a bootloader makes it possible for the user to upload a patched version of the firmware onto the micro. The bootloader receives information from a communication line (a USB port, serial port, ethernet port, WiFi connection, etc...) and stores this data into program memory. Once the full firmware has been received, the micro can happily run its updated code.

There is one big security issue to worry about with bootloaders. A company may want to stop their customers from writing their own firmware and uploading it onto the micro. For example, this might be for protection reasons - hackers might be able to access parts of the device that weren't meant to be accessed. One way of stopping this is to add encryption. The company can add their own secret signature to the firmware code and encrypt it with a secret key. Then, the bootloader can decrypt the incoming firmware and confirm that the incoming firmware is correctly signed. Users will not know the secret key or the signature tied to the firmware, so they won't be able to "fake" their own.

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'

In [ ]:
%%bash -s "$PLATFORM" 
cd ../../../hardware/victims/firmware/bootloader-aes256
make PLATFORM=$1 CRYPTO_TARGET=NONE

In [ ]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
fw_path = "../../../hardware/victims/firmware/bootloader-aes256/bootloader-aes256-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

## The Situation

Simply put, we've got a target device running an encrypted bootloader (a program used to upload new code onto a device) and we want to see if we can get our own code running on the device. We've done a bit of sniffing on the serial lines when the device's firmware is being updated and we've learned the following:

* The device communicates over serial at 38400bps
* When writing memory, the first byte is always zero (probably a command byte)
* There's a 16 byte block of random looking memory (aka it doesn't look like firmware). This part is probably encrypted
* There's a 2 byte CRC at the end of each message
* There's no repetition in the ciphertext.

All together this looks like:

```
       |<-------- Encrypted block (16 bytes) ---------->|
       |                                                |
+------+------+------+------+------+------+ .... +------+------+------+
| 0x00 |              Random looking data               |   CRC-16    |
+------+------+------+------+------+------+ .... +------+------+------+
```

After sending data to the bootloader it responds with either `0xA4` or `0xA1`. The former only happened when we sent a bad CRC.

This time, we won't be triggering off of our trigger pins (you can remove them from the code if you'd like).

From our initial sniffing of the communication lines, we've got the first few messages that were sent:

In [ ]:
import pickle
with open("./firmware.pickle", "rb") as f:
    encrypted_firmware = pickle.load(f)

## Doing Recon

Our first step will be to see if we can learning anything about the bootloader from looking at its power traces. Let's start with the boot sequence:

In [ ]:
scope.trigger.triggers = "nrst"

In [ ]:
scope.adc.samples = 24400

In [ ]:
scope.arm()
reset_target(scope)
scope.capture()
wave = scope.get_last_trace()

In [ ]:
cw.plot(wave)

The device does appear to be doing something, but it's clearly nothing major - no encryptions or anything. Every microcontroller has boot code and operations that run when it's reset. The device may even have its own bootloader running in ROM! Let's move onto the messages themselves. We do know that there's a CRC for data integrety. We can use the following code to calculate the CRC for us:

In [ ]:
# Class Crc
#############################################################
# These CRC routines are copy-pasted from pycrc, which are:
# Copyright (c) 2006-2013 Thomas Pircher <tehpeh@gmx.net>
#
class Crc(object):
    """
    A base class for CRC routines.
    """

    def __init__(self, width, poly):
        """The Crc constructor.

        The parameters are as follows:
            width
            poly
            reflect_in
            xor_in
            reflect_out
            xor_out
        """
        self.Width = width
        self.Poly = poly


        self.MSB_Mask = 0x1 << (self.Width - 1)
        self.Mask = ((self.MSB_Mask - 1) << 1) | 1

        self.XorIn = 0x0000
        self.XorOut = 0x0000

        self.DirectInit = self.XorIn
        self.NonDirectInit = self.__get_nondirect_init(self.XorIn)
        if self.Width < 8:
            self.CrcShift = 8 - self.Width
        else:
            self.CrcShift = 0

    def __get_nondirect_init(self, init):
        """
        return the non-direct init if the direct algorithm has been selected.
        """
        crc = init
        for i in range(self.Width):
            bit = crc & 0x01
            if bit:
                crc ^= self.Poly
            crc >>= 1
            if bit:
                crc |= self.MSB_Mask
        return crc & self.Mask


    def bit_by_bit(self, in_data):
        """
        Classic simple and slow CRC implementation.  This function iterates bit
        by bit over the augmented input message and returns the calculated CRC
        value at the end.
        """
        # If the input data is a string, convert to bytes.
        if isinstance(in_data, str):
            in_data = [ord(c) for c in in_data]

        register = self.NonDirectInit
        for octet in in_data:
            for i in range(8):
                topbit = register & self.MSB_Mask
                register = ((register << 1) & self.Mask) | ((octet >> (7 - i)) & 0x01)
                if topbit:
                    register ^= self.Poly

        for i in range(self.Width):
            topbit = register & self.MSB_Mask
            register = ((register << 1) & self.Mask)
            if topbit:
                register ^= self.Poly

        return register ^ self.XorOut
    
bl_crc = Crc(width = 16, poly=0x1021)

Let's definte a function to do the communication and capture a trace for us. We can try triggering off of our message. There's not much memory on the target, so it's probably decryption on the fly instead of reading in a whole bunch of memory, then doing the decryption.

In [ ]:
scope.trigger.triggers = "tio2"
scope.adc.samples = 24400
scope.adc.decimate = ??? # try to get the full encryption in a single trace, then set back to 1
scope.adc.offset = 0
def cap_trace(enc_block):
    message = [0x00]
    target.read()

    key, text = ktp.next()
    message.extend(enc_block)

    crc = bl_crc.bit_by_bit(enc_block)
    message.append(crc >> 8)
    message.append(crc & 0xFF)

    

    target.write(message[:-1])
    time.sleep(0.01)
    scope.arm()
    target.write([crc&0xFF])
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')
    response = target.read()
    if response:
        if ord(response[0]) != 0xA4:
            # Bad response, just skip
            #print("Bad response: {:02X}".format(ord(response[0])))
            return None

    return scope.get_last_trace()
    
ktp = cw.ktp.Basic()
text, key = ktp.next()
wave = cap_trace(text)
wave2 = cap_trace(text)

In [ ]:
cw.plot(wave) * cw.plot(wave2)

We've found the decryption. Some things to notice from this:

1. The target immediately goes from reading the ciphertext to running a decryption - there's no preprocessing done at all. This means the ciphertext is likely being immediately fed into the decryption algorithm.
1. We can see operations in the encryption being repeated. This looks a lot like AES - we can see a long distinct operation that's probably MixColumns in there and the overall structure looks a lot like what we saw when looking at TINYAES128C encryptions. Let's make a weak assumption that this is AES128 - we can adjust this as we learn more about the bootloader.
1. We can't see the full encryption
1. There's some jitter here. We'll probably have to resync the traces if we run a CPA attack

Since we can't see the full encryption, decimate the ADC (we don't care too much about the fine details here) and take another look...


## The full encryption

You should see that instead of 9 repititions of MixColumns (or what we assume is MixColumns), there's actually 13. This rules out AES128, but AES256 actually has 14 rounds! We can still attack AES256 without much issue: we basically just have to run two CPA attacks, one for the first half of the key and another for the second half. Again, we're not 100% about this, but it's a good starting point. 

As we mentioned in the debriefing about the bootloader, the ciphertext never seems to repeat. It's pretty unlikely that this device is using straight AES256 (if it even is using AES256), since firmware usually has blocks that repeat. More likely is that AES is being used as a stream cipher: https://en.wikipedia.org/wiki/Block_cipher_mode_of_operation. This could pose an immediate problem for our attack efforts: we need to know either what's going into or coming out of the AES block to perform a CPA attack. However, some of the modes listed on that page (if it's even using one on that page) feed an IV or a counter into that block instead of the plaintext or the ciphertext. We didn't see any encryption operations happening on startup (which the device could've done if it was using one of these IV/counter modes), so we'll probably be okay with a normal CPA attack. Let's try it and see if we can get anything out of it:

In [ ]:
from tqdm.notebook import trange
project = cw.create_project("projects/Tutorial_A5", overwrite=True)
scope.adc.offset = 0
scope.adc.decimate = 1
for i in trange(100):
    ktp = cw.ktp.Basic()
    key, text = ktp.next()
    wave = cap_trace(text)
    trace = cw.Trace(wave, text, bytearray([0]*16), bytearray([0]*16))
    project.traces.append(trace)

In [ ]:
cw.plot(project.waves[0])

We can eliminate a lot of this jitter by using the resync SAD module:

In [ ]:
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa

leak_model = cwa.leakage_models.inverse_sbox_output
resync = cwa.preprocessing.ResyncSAD(project)
resync.enabled=True
resync.ref_trace = 0
resync.target_window = (???, ???)
resync.max_shift = 7000
new_proj = resync.preprocess()



Check to make sure your traces are resynced:

In [ ]:
plt = cw.plot([])
for i in range(10):
    plt *= cw.plot(new_proj.waves[i])

plt

All that's left is to actually run the attack. We don't know the correct key, so it won't be highlighted in red.

In [ ]:
attack = cwa.cpa(new_proj, leak_model)
#attack.pont_range = [???, ???]

#key = [0xea, 0x79, 0x79, 0x20, 0xc8, 0x71, 0x44, 0x7d, 0x46, 0x62, 0x5f, 0x51, 0x85, 0xc1, 0x3b, 0xcb]

cb = cwa.get_jupyter_callback(attack)
attack_results = attack.run(cb)

The difference in correlation between the best key guess and the next best one makes this look very promising! We now know we're correct about two things:

1. The bootloader is actually decrypting the ciphertext
1. The device is using AES

In [ ]:
calc_round_key = attack_results.key_guess()

With that done, we now need to get the second half of the key. Pop over to [Extending AES-128 Attacks to AES-256](Extending%20AES-128%20Attacks%20to%20AES-256.ipynb), since that page explains how to do that...

Back? Let's go through and see if that theory actually holds up.

To make a new model, we start off by inheriting the `AESLeakageHelper` class. We need to make a `leakage()` method that calculates the Hamming weight we use in the CPA attack. To get you started:

In [ ]:
import chipwhisperer as cw
class AES256_Round13_Model(cwa.AESLeakageHelper):
    def leakage(self, pt, ct, guess, bnum):
        #You must put YOUR recovered 14th round key here - this example may not be accurate!
        calc_round_key = [0xea, 0x79, 0x79, 0x20, 0xc8, 0x71, 0x44, 0x7d, 0x46, 0x62, 0x5f, 0x51, 0x85, 0xc1, 0x3b, 0xcb]
        state = reverse_round_14(self, pt, calc_round_key)
        state = reverse_round_13(self, state) #reverse state just before inv_subbytes
        return self.inv_sbox(state[bnum] ^ guess[bnum])

Now we just need to make the `reverse_round_14()` and `reverse_round_13()` functions. By passing the class in, we get access to `self.inv_shiftrows()`, `self.inv_subbytes()`, and `self.inv_mixcolumns()`.

In [ ]:
def reverse_round_14(self, pt, key):
    state = [pt[i] ^ key[i] for i in range(16)] #AddRoundKey
    state = ???
    state = ???
    return state # we're now at the end of decryption round 1

def reverse_round_13(self, state):
    state = ???
    state = ???
    return state

From there, just update the leakage model and rerun the attack:

In [ ]:
leak_model = cwa.leakage_models.new_model(AES256_Round13_Model)
attack.leak_model = leak_model
cb = cwa.get_jupyter_callback(attack)
attack_results = attack.run(cb)

If you built your model correctly, you should again see a pretty likely guess for a key. All that's left now is to combine the 14th and 13th round keys, then use that to figure out the 0th and 1st round keys.

We'll start by getting the transformed 13th round key out of the attack results:

In [ ]:
#calc_round_key = [0xea, 0x79, 0x79, 0x20, 0xc8, 0x71, 0x44, 0x7d, 0x46, 0x62, 0x5f, 0x51, 0x85, 0xc1, 0x3b, 0xcb]
rec_key = calc_round_key

In [ ]:
rec_key2 = []
for bnum in attack_results.find_maximums():
    print("Best Guess = 0x{:02X}, Corr = {}".format(bnum[0][0], bnum[0][2]))
    rec_key2.append(bnum[0][0])

Now we need to transform that key into the actual 13th round key by running it through ShiftRows and MixColumns:

In [ ]:
real_key2 = cwa.aes_funcs.shiftrows(rec_key2)
real_key2 = cwa.aes_funcs.mixcolumns(real_key2)

print("Recovered:", end="")
for subkey in real_key2:
    print(" {:02X}".format(subkey), end="")
print("")

then we can combine the keys:

In [ ]:
rec_key_comb = real_key2.copy()
rec_key_comb.extend(rec_key)

print("Key:", end="")
for subkey in rec_key_comb:
    print(" {:02X}".format(subkey), end="")
print("")

and use ChipWhisperer's built in key scheduler to reverse them to the 0th and 1st round keys:

In [ ]:
btldr_key = leak_model.key_schedule_rounds(rec_key_comb, 13, 0)
btldr_key.extend(leak_model.key_schedule_rounds(rec_key_comb, 13, 1))
print("Key:", end="")
for subkey in btldr_key:
    print(" {:02X}".format(subkey), end="")
print("")

So we were clearly right about the bootloader running AES256! However, if we try decrypting some of our firmware with the encryption key:

In [ ]:
from Crypto.Cipher import AES
cipher = AES.new(bytes(btldr_key), AES.MODE_ECB)
print(bytearray(cipher.decrypt(encrypted_firmware[:16])))
print(bytearray(cipher.decrypt(encrypted_firmware[16:32])))

As we expected, we still get gibberish out of this - the target is definitely using AES as a stream cipher. The question now is, which block cipher mode is it using? Well, we know the ciphertext is being decrypted. We can narrow it down a bit by looking at the end of the encryption. Increase the offset until you reach the end of the encryption.

In [ ]:
#scope.adc.offset = 50000
#wave = cap_trace(text)
#key, text = ktp.next()
wave = cap_trace(text)

In [ ]:
%matplotlib notebook
import matplotlib.pyplot as plt
plt.figure()
plt.plot(wave)
plt.show()

It might be hard to pick out, but you should be able to find 16 XOR short operations just after the end of the last round of AES. It might be using CBC mode, which means it'll be using the ciphertext as part of the encryption and decryption of subsequent blocks. Let's do a quick check:

In [ ]:
cipher = AES.new(bytes(btldr_key), AES.MODE_ECB)
dec_fw = cipher.decrypt(encrypted_firmware[16:32])
fw = [dec_fw[i] ^ encrypted_firmware[i] for i in range(16)]

In [ ]:
print(fw)

It looks like we're finally getting some valid output. Let's try the next block:

In [ ]:
cipher = AES.new(bytes(btldr_key), AES.MODE_ECB)
dec_fw = cipher.decrypt(encrypted_firmware[32:48])
fw = [dec_fw[i] ^ encrypted_firmware[i+16] for i in range(16)]
print(fw)

This is very promising! This firmware is repeated twice. The all `FFs` is probably just empty flash memory. The beginning is more curious though:

* It's probably not a flash constant - otherwise it wouldn't be in two blocks in a row
* That's only enough room for a few instructions at most. Again, it's a little strange that it would be repeated like this

It might be some sort of signature. After all, the device doesn't want to write anything to memory unless the ciphertext has properly been decrypted.

There's still the issue of the first block of memory though. There's another secret value called an initialization vector that we need to decrypt that block. To be able to recover that, we'll need to revisit the DPA attack:

### Attack Theory

The bootloader applies the IV to the AES decryption result (`DR`) by calculating


$\text{PT} = \text{DR} \oplus \text{IV}$

where DR is the decrypted ciphertext, IV is the secret vector, and PT is the plaintext that the bootloader will use later. We only have access to one of these: since we know the AES-256 key, we can calculate DR. This exclusive or will be visible in the power traces.

This is enough information for us to attack a single bit of the IV. Suppose we only wanted to get the first bit (bit 0) of the first byte (byte 0) of the IV. We could do the following:

* Split all of the traces into two groups: those with `(DR[0] & 0x01) = 0`, and those with `(DR[0] & 0x01) = 1`. 
* Calculate the average trace for both groups.
* Find the difference between the two averages. Provided we've got sufficient data, we should see a spike where the xor is occuring.
* Look at the direction of the spike to decide if the IV bit is 0 `(PT[0] = DR[0])` or if the IV bit is 1 `(PT[0] = ~DR[0])`.

This is effectively a DPA attack on a single bit of the IV. We can repeat this attack across the whole IV by instead separating by `(DR[byte] & (1 << bit) = 0` and `(DR[byte] & (1 << bit) = bit`.

We'll need to reset the device every encryption since it only uses the IV in the first encryption. This leads to a slightly modified capture loop. You'll need to adjust your offset since we're now triggering near the beginning of the UART transmit instead of nera the end. Run the loop, then interrupt it to get a wave. Then plot and adjust your offset. Repeat until you're near the end of the encryption again:

In [ ]:
from Crypto.Cipher import AES
import numpy as np

from tqdm.notebook import trange
import numpy as np
import time
traces = []
keys = []
plaintexts = []

from tqdm.notebook import trange
project = cw.create_project("projects/Tutorial_A5_IV", overwrite=True)
scope.adc.decimate = 1
scope.adc.offset = 51000
scope.adc.timeout = 1
scope.trigger.triggers = "tio2"
for i in trange(1000):
    scope.io.nrst = 0
    time.sleep(0.02)
    scope.io.nrst = "high_z"
    time.sleep(0.01)
    okay = 0
    while not okay:
        target.write('\0xxxxxxxxxxxxxxxxxx')
        time.sleep(0.005)
        response = target.read()
        i += 1
        if response:
            if ord(response[0]) == 0xA1:
                okay = 1
    message = [0x00]
    
    target.flush()
    
    key, text = ktp.new_pair()  # manual creation of a key, text pair can be substituted here
    
    wave = cap_trace(text)
    if wave is None:
        continue
    
    
    #wave = scope.get_last_trace()
    trace = cw.Trace(wave, text, bytearray([0]*16), bytearray([0]*16))
    project.traces.append(trace)

As you can see, we'll again need to resync to get rid of the jitter:

In [ ]:
plt = cw.plot([])
for i in range(2):
    plt *= cw.plot(project.waves[i])
    
plt

Adjust the target window here as necessary:

In [ ]:
import chipwhisperer as cw
import chipwhisperer.analyzer as cwa

leak_model = cwa.leakage_models.inverse_sbox_output
resync = cwa.preprocessing.ResyncSAD(project)
resync.enabled=True
resync.ref_trace = 0
resync.target_window = (???, ???)
resync.max_shift = 6000
new_proj = resync.preprocess()

In [ ]:
plt = cw.plot([])
for i in range(10):
    plt *= cw.plot(new_proj.waves[i])
    
plt

Some numpy functions will be useful here, so we'll convert our ChipWhisperer project to numpy arrays:

In [ ]:
trace_array = np.array([new_proj.waves[i] for i in range(len(new_proj.traces))])
textin_array = np.array([new_proj.textins[i] for i in range(len(new_proj.traces))])

We need to decrypt what we sent to the device to get one half of the input to the XOR:

In [ ]:
knownkey = [0x94, 0x28, 0x5D, 0x4D, 0x6D, 0xCF, 0xEC, 0x08, 0xD8, 0xAC, 0xDD, 0xF6, 0xBE, 0x25, 0xA4, 0x99,
            0xC4, 0xD9, 0xD0, 0x1E, 0xC3, 0x40, 0x7E, 0xD7, 0xD5, 0x28, 0xD4, 0x09, 0xE9, 0xF0, 0x88, 0xA1]

knownkey = bytes(knownkey)
dr = []
aes = AES.new(knownkey, AES.MODE_ECB)
for i in range(len(new_proj.traces)):
    ct = bytes(textin_array[i])
    pt = aes.decrypt(ct)
    d = [bytearray(pt)[i] for i in range(16)]
    dr.append(d)

The basic idea is the same as the DPA attack: guess a bit and group traces based on that. We'll do the first byte here as an example. For each of the bits, you should see roughly half the traces fall into each group:

In [ ]:
grouped_byte_traces = []
byte = ??? # start with zero, then come back later and change

for bit in range(8):
    grouped_bit_traces = [], []
    for i in range(len(new_proj.traces)):
        if (dr[i][byte] & (1 << bit)):
            grouped_bit_traces[0].append(trace_array[i])
        else:
            grouped_bit_traces[1].append(trace_array[i])
    grouped_byte_traces.append(grouped_bit_traces)
    print(len(grouped_bit_traces[0]))

SyntaxError: invalid syntax (<ipython-input-1-b8a799d34f4a>, line 2)

Now we need to figure out where the XOR operation is happening. You'll need to use the shape of the plot and the following plot of the difference of means for each bit. Each colour represents a different bit. For the correct part on the plot, you should see a distinct separation, with some bits peaking above zero, and others peaking below zero. If you see some colours in between peaks, it is probably not the right location. Repeat this for a few bytes - the location should change. Note down this change, as you'll have to use it to adjust the analysis location later:

In [ ]:
# Find averages and differences
diffs = []
for i in range(8):
    means = np.average(grouped_byte_traces[i][0], axis=0), np.average(grouped_byte_traces[i][1], axis=0)
    diffs.append(means[1] - means[0])

In [ ]:
# Split traces into 2 groups
from bokeh.plotting import figure, show
from bokeh.io import output_notebook

output_notebook()
p = figure()

xrange = range(len(diffs[0]))
xrange2 = range(len(trace_array[0]))
colours = ["red", "blue", "green", "black"]
plt = cw.plot([])
for i in range(8):
    plt *= cw.plot(diffs[i]).opts(color=colours[i%4])
plt

Fill in the position of the XOR, as well as how much it changes for each byte. All we're doing here is going byte by byte and bit by bit, and seeing if the difference in means is greater than or less than zero:

In [ ]:
btldr_IV = [0] * 16 #181
for byte in range(16):
    location = ??? + byte * ???
    iv = 0
    for bit in range(8):
        pt_bits = [((dr[i][byte] >> (7-bit)) & 0x01) for i in range(len(new_proj.traces))]

        # Split traces into 2 groups
        groupedPoints = [[] for _ in range(2)]
        for i in range(len(new_proj.traces)):
            groupedPoints[pt_bits[i]].append(trace_array[i][location])
            
        means = []
        for i in range(2):
            means.append(np.average(groupedPoints[i]))
        diff = means[1] - means[0]
        
        iv_bit = 1 if diff > 0 else 0
        iv = (iv << 1) | iv_bit
        
        print(iv_bit, end = " ")
        
    print("{:02X}".format(iv))
    btldr_IV[byte] = iv
    
print(btldr_IV)

Finally, we can do the full decryption!

In [ ]:
cipher = AES.new(bytes(btldr_key), AES.MODE_ECB)
first_pt = cipher.decrypt(encrypted_firmware[:16])
first_pt = [first_pt[i] ^ btldr_IV[i] for i in range(16)]
print(bytearray(first_pt))

As you can see, the first line of firmware is `deadbeefaabbccddeeff0011`